In [1]:
import pandas as pd
import numpy as np
import re

# Load raw data
df = pd.read_csv('../csvs/raw/nascar_2025_2026.csv')  # my scraped file
print(f"Loaded {len(df)} rows")
print(f"Original columns: {df.columns.tolist()}")

Loaded 2013 rows
Original columns: ['year', 'race_number', 'race_name', 'race_date', 'track', 'track_type', 'track_miles', 'total_laps', 'caution_flags', 'caution_laps', 'lead_changes', 'avg_speed_mph', 'pole_speed_mph', 'margin_of_victory', 'attendance', 'finish_pos', 'start_pos', 'car_number', 'driver', 'sponsor', 'owner', 'car_make', 'laps_completed', 'status', 'laps_led', 'points', 'loop_avg_speed', 'loop_green_flag_passes', 'loop_quality_passes', 'loop_fastest_lap', 'loop_driver_rating']


In [2]:
# Create a column mapping for consistent naming
# Adjust based on your actual column names
column_mapping = {
    # Common variations -> Standard names
    'Fin': 'Finish_Position',
    'finish_pos': 'Finish_Position',
    'fin': 'Finish_Position',
    'Pos': 'Finish_Position',
    'Position': 'Finish_Position',
    'St': 'Start_Position',
    'start_pos': 'Start_Position',
    'Start': 'Start_Position',
    'driver': 'Driver',
    'Team': 'Team',
    'owner': 'Owner',
    'sponsor': 'Sponsor',
    'points': 'Points',
    'race_date': 'Race_Date',
    'Laps': 'Laps_Completed',
    'Led': 'Laps_Led',
    'laps_led': 'Laps_Led',
    'laps_completed': 'Laps_Completed',
    'Race': 'Race_Name',
    'Track': 'Track',
    'Date': 'Race_Date'
}

# Rename columns that exist in our mapping
df = df.rename(columns={col: column_mapping[col]
                        for col in df.columns
                        if col in column_mapping})

print(f"Standardized columns: {df.columns.tolist()}")

Standardized columns: ['year', 'race_number', 'race_name', 'Race_Date', 'track', 'track_type', 'track_miles', 'total_laps', 'caution_flags', 'caution_laps', 'lead_changes', 'avg_speed_mph', 'pole_speed_mph', 'margin_of_victory', 'attendance', 'Finish_Position', 'Start_Position', 'car_number', 'Driver', 'Sponsor', 'Owner', 'car_make', 'Laps_Completed', 'status', 'Laps_Led', 'Points', 'loop_avg_speed', 'loop_green_flag_passes', 'loop_quality_passes', 'loop_fastest_lap', 'loop_driver_rating']


In [3]:
# Convert Finish_Position to numeric
# Handle non-numeric values (DNF, DNS, DQ may appear as text)
def clean_position(pos)-> int:
    """Convert position to integer, handling special cases."""
    if pd.isna(pos):
        return np.nan
    pos_str = str(pos).strip()
    # Extract just the number
    match = re.search(r'\d+', pos_str)
    if match:
        return int(match.group())
    return np.nan

df['Finish_Position'] = df['Finish_Position'].apply(clean_position)

# Same for Start_Position
if 'Start_Position' in df.columns:
    df['Start_Position'] = df['Start_Position'].apply(clean_position)
    df['Start_Position'] = pd.to_numeric(df['Start_Position'], errors='coerce').astype('Int64')
if 'Points' in df.columns:
    df['Points'] = pd.to_numeric(df['Points'], errors='coerce').astype('Int64')

# Convert Laps_Led to numeric
if 'Laps_Led' in df.columns:
    df['Laps_Led'] = pd.to_numeric(df['Laps_Led'], errors='coerce').fillna(0).astype(int)

# Convert Race_Date to datetime
if 'Race_Date' in df.columns:
    df['Race_Date'] = pd.to_datetime(df['Race_Date'], errors='coerce')

print("\nData types after conversion:")
print(df.dtypes)


Data types after conversion:
year                               int64
race_number                        int64
race_name                            str
Race_Date                 datetime64[us]
track                                str
track_type                           str
track_miles                      float64
total_laps                       float64
caution_flags                      int64
caution_laps                       int64
lead_changes                       int64
avg_speed_mph                    float64
pole_speed_mph                   float64
margin_of_victory                    str
attendance                       float64
Finish_Position                    int64
Start_Position                     Int64
car_number                         int64
Driver                               str
Sponsor                              str
Owner                                str
car_make                             str
Laps_Completed                     int64
status                     

In [4]:
# View unique driver names to identify inconsistencies
print("Unique drivers:", df['Driver'].nunique())
print("\nSample driver names:")
for name in sorted(df['Driver'].dropna().unique())[:20]:
    print(f"  '{name}'")
num_missing=(df['Driver'].isna().sum())
print(f"Missing {num_missing} drivers")

Unique drivers: 62

Sample driver names:
  'A.J. Allmendinger'
  'Alex Bowman'
  'Anthony Alfredo'
  'Austin Cindric'
  'Austin Dillon'
  'Austin Hill'
  'B.J. McLeod'
  'Brad Keselowski'
  'Brennan Poole'
  'Bubba Wallace'
  'Burt Myers'
  'Carson Hocevar'
  'Casey Mears'
  'Chad Finchum'
  'Chase Briscoe'
  'Chase Elliott'
  'Chris Buescher'
  'Christopher Bell'
  'Cody Ware'
  'Cole Custer'
Missing 0 drivers


In [5]:
def clean_driver_name(name):
    """Standardize driver name format."""
    if pd.isna(name):
        return name

    # Remove extra whitespace
    name = ' '.join(str(name).split())

    # Remove common suffixes/prefixes
    name = re.sub(r'\s*\(i\)|\s*\*|\s*#\d+', '', name)

    # Capitalize properly
    name = name.title()

    return name.strip()

df['Driver'] = df['Driver'].apply(clean_driver_name)

# Check for common variations and standardize
driver_corrections = {
    # Add corrections as you find them, e.g.:
    # 'Denny M Hamlin': 'Denny Hamlin',
    # 'Hamlin, Denny': 'Denny Hamlin',
}

df['Driver'] = df['Driver'].replace(driver_corrections)

print(f"\nUnique drivers after cleaning: {df['Driver'].nunique()}")


Unique drivers after cleaning: 62


In [6]:
def clean_team_name(name):
    """Standardize team name format."""
    if pd.isna(name):
        return name

    name = ' '.join(str(name).split())

    # Common standardizations
    corrections = {
        'Joe Gibbs Racing': 'Joe Gibbs Racing',
        'JGR': 'Joe Gibbs Racing',
        'Hendrick Motorsports': 'Hendrick Motorsports',
        'HMS': 'Hendrick Motorsports',
        'Team Penske': 'Team Penske',
        'Penske': 'Team Penske',
        'Stewart-Haas Racing': 'Stewart-Haas Racing',
        'SHR': 'Stewart-Haas Racing',
        '23XI Racing': '23XI Racing',
        '23XI': '23XI Racing',
        'Richard Childress Racing': 'Richard Childress Racing',
        'RCR': 'Richard Childress Racing',
        'Trackhouse Racing': 'Trackhouse Racing',
    }

    for old, new in corrections.items():
        if old.lower() in name.lower():
            return new

    return name.strip()

df['Owner'] = df['Owner'].apply(clean_team_name)

print(f"Unique teams: {df['Owner'].nunique()}")
print("\nTeam list:")
for team in (df['Owner'].sort_values().unique()):
    print(f"  {team}")

Unique teams: 20

Team list:
  23XI Racing
  B.J. McLeod
  Beard Motorsports
  Bob Jenkins
  Carl Long
  Gene Haas
  HYAK Motorsports
  JR Motorsports
  Jack Roush
  Joe Gibbs
  Legacy Motor Club
  Matthew Kaulig
  Richard Childress
  Rick Hendrick
  Rick Ware
  Spire Motorsports
  Team AmeriVet
  Team Penske
  Trackhouse Racing
  Wood Brothers
  nan


In [7]:
# Create sponsor mapping for your target sponsors
# This maps driver+team combinations to their primary sponsor
# Based on your Project 1 selections

# FIX (2026-08-16): the previous version of this cell mapped each driver/team
# pair to ONE sponsor for their entire season and counted every race they ran
# as that sponsor's race. Checking the real per-race `Sponsor` column (scraped
# from racing-reference.info's race-by-race results, not inferred) showed this
# was wrong for all five sponsors, not just Cheddar's — most Cup cars sell the
# primary paint scheme race-by-race, so the "anchor" sponsor most fans
# associate with a driver is often the actual on-car sponsor for well under
# half the season:
#   Progressive/Denny Hamlin:        20 of 52 races (38%)
#   Castrol/Brad Keselowski:         11 of 53 races (21%)
#   Busch Light/Ross Chastain:       25 of 53 races (47%)
#   Love's Travel Stops/Todd Gilliland: 29 of 53 races (55%)
#   Cheddar's Scratch Kitchen/Kyle Busch: 12 of 48 races (25%)
# Now a race only counts for a sponsor if that sponsor's name (or a livery
# variant of it, e.g. "Busch Light Apple", "Cheddar's Patriotic", "Progressive
# Insurance") actually appears in that race's Sponsor field for that
# driver/team. This shrinks every sponsor's sample size but makes "visibility"
# mean "races that sponsor actually paid to be on the car," not "races this
# driver happened to run."

sponsor_mapping = {
    # Format: ('Driver Name', 'Team Name'): ('Primary Sponsor', 'match keyword')
    ('Denny Hamlin', 'Joe Gibbs'): ('Progressive', 'progressive'),
    ('Kyle Busch', 'Richard Childress'): ('Cheddar\'s Scratch Kitchen', 'cheddar'),
    ('Todd Gilliland', 'Bob Jenkins'): ('Love\'s Travel Stops', 'love'),
    ('Brad Keselowski', 'Jack Roush'): ('Castrol', 'castrol'),
    ('Ross Chastain', 'Trackhouse Racing'): ('Busch Light', 'busch'),
}

def get_sponsor(row):
    """Look up primary sponsor for driver/team combination, but only credit
    the race to that sponsor if the actual per-race Sponsor field matches —
    otherwise this was someone else's primary slot that week."""
    key = (row['Driver'], row['Owner'])
    if key not in sponsor_mapping:
        return 'Other/Unknown'
    sponsor_name, keyword = sponsor_mapping[key]
    race_sponsor_text = str(row.get('Sponsor', ''))
    if keyword.lower() in race_sponsor_text.lower():
        return sponsor_name
    return 'Other/Unknown'

df['Primary_Sponsor'] = df.apply(get_sponsor, axis=1)

# Check sponsor distribution
print("Sponsor distribution:")
print(df['Primary_Sponsor'].value_counts())

Sponsor distribution:
Primary_Sponsor
Other/Unknown                1916
Love's Travel Stops            29
Busch Light                    25
Progressive                    20
Cheddar's Scratch Kitchen      12
Castrol                        11
Name: count, dtype: int64


In [8]:
# Define your target sponsors
target_sponsors = [
    'FedEx',
    'Castrol',
    'Love\'s Travel Stops',
    'Busch Light',
    'Cheddar\'s Scratch Kitchen',
    
]

# Create filtered dataset with just target sponsors
df_targets = df[df['Primary_Sponsor'].isin(target_sponsors)].copy()

print(f"\nFiltered to target sponsors: {len(df_targets)} rows")
print(df_targets['Primary_Sponsor'].value_counts())


Filtered to target sponsors: 77 rows
Primary_Sponsor
Love's Travel Stops          29
Busch Light                  25
Cheddar's Scratch Kitchen    12
Castrol                      11
Name: count, dtype: int64


In [9]:
# Add race number for time series analysis
races_order = df[['year', 'race_number']].drop_duplicates().sort_values(['year', 'race_number'])
race_number_map = {(row.year, row.race_number): i + 1 for i, row in enumerate(races_order.itertuples())}
df['race_number'] = df.apply(lambda r: race_number_map[(r['year'], r['race_number'])], axis=1)

# Calculate positions gained/lost
if 'Start_Position' in df.columns:
    df['Positions_Gained'] = df['Start_Position'] - df['Finish_Position']

print("Added derived columns:")
print(df[['race_number', 'Driver', 'Start_Position',
          'Finish_Position', 'Positions_Gained']].head(10))

Added derived columns:
   race_number            Driver  Start_Position  Finish_Position  \
0            1     William Byron               5                1   
1            1     Tyler Reddick              11                2   
2            1    Jimmie Johnson              40                3   
3            1     Chase Briscoe               1                4   
4            1  John H. Nemechek              18                5   
5            1       Alex Bowman              38                6   
6            1       Ryan Blaney              16                7   
7            1    Austin Cindric               2                8   
8            1   Justin Allgaier              19                9   
9            1    Chris Buescher               6               10   

   Positions_Gained  
0                 4  
1                 9  
2                37  
3                -3  
4                13  
5                32  
6                 9  
7                -6  
8                10

In [10]:
# Calculate season-level performance metrics per driver
driver_stats = df.groupby(['Driver', 'Owner', 'Primary_Sponsor']).agg({
    'Finish_Position': ['mean', 'min', 'std', 'count'],
    'Laps_Led': ['sum', 'mean'],
    'race_number': 'nunique'
}).round(2)

# Flatten column names
driver_stats.columns = [
    'Avg_Finish', 'Best_Finish', 'Finish_StdDev', 'Races_Started',
    'Total_Laps_Led', 'Avg_Laps_Led_Per_Race', 'Races_Completed'
]

driver_stats = driver_stats.reset_index()

# Add calculated metrics
# Top 5 finishes
top5 = df[df['Finish_Position'] <= 5].groupby('Driver').size()
driver_stats['Top_5_Finishes'] = driver_stats['Driver'].map(top5).fillna(0).astype(int)

# Top 10 finishes
top10 = df[df['Finish_Position'] <= 10].groupby('Driver').size()
driver_stats['Top_10_Finishes'] = driver_stats['Driver'].map(top10).fillna(0).astype(int)

# Wins
wins = df[df['Finish_Position'] == 1].groupby('Driver').size()
driver_stats['Wins'] = driver_stats['Driver'].map(wins).fillna(0).astype(int)

# Calculate rates
driver_stats['Top_5_Rate'] = (driver_stats['Top_5_Finishes'] /
                               driver_stats['Races_Started'] * 100).round(1)
driver_stats['Top_10_Rate'] = (driver_stats['Top_10_Finishes'] /
                                driver_stats['Races_Started'] * 100).round(1)

print("\n=== Driver Season Statistics ===")
print(driver_stats.sort_values('Avg_Finish'))


=== Driver Season Statistics ===
               Driver              Owner Primary_Sponsor  Avg_Finish  \
30       Denny Hamlin          Joe Gibbs     Progressive       10.60   
64      Tyler Reddick        23XI Racing   Other/Unknown       12.47   
17      Chase Elliott      Rick Hendrick   Other/Unknown       12.74   
29       Denny Hamlin          Joe Gibbs   Other/Unknown       13.00   
19   Christopher Bell          Joe Gibbs   Other/Unknown       13.53   
..                ...                ...             ...         ...   
13        Casey Mears  Beard Motorsports   Other/Unknown       36.00   
59         Timmy Hill          Carl Long   Other/Unknown       36.00   
36          Joey Gase          Carl Long   Other/Unknown       37.00   
65         Will Brown     Matthew Kaulig   Other/Unknown       39.00   
32  Helio Castroneves  Trackhouse Racing   Other/Unknown       39.00   

    Best_Finish  Finish_StdDev  Races_Started  Total_Laps_Led  \
30            1          11.92      

In [11]:
# Aggregate to sponsor level
sponsor_stats = df.groupby('Primary_Sponsor').agg({
    'Finish_Position': ['mean', 'min', 'std'],
    'Laps_Led': ['sum', 'mean'],
    'Driver': 'nunique',
    'race_number': 'count'
}).round(2)

sponsor_stats.columns = [
    'Avg_Finish', 'Best_Finish', 'Finish_StdDev',
    'Total_Laps_Led', 'Avg_Laps_Led',
    'Num_Drivers', 'Total_Race_Entries'
]

sponsor_stats = sponsor_stats.reset_index()

# Add sponsor-level rates
for sponsor in sponsor_stats['Primary_Sponsor']:
    sponsor_df = df[df['Primary_Sponsor'] == sponsor]
    total_races = len(sponsor_df)

    idx = sponsor_stats['Primary_Sponsor'] == sponsor
    sponsor_stats.loc[idx, 'Top_5_Rate'] = (
        len(sponsor_df[sponsor_df['Finish_Position'] <= 5]) / total_races * 100
    )#.round(1)
    sponsor_stats.loc[idx, 'Top_10_Rate'] = (
        len(sponsor_df[sponsor_df['Finish_Position'] <= 10]) / total_races * 100
    )#.round(1)
    sponsor_stats.loc[idx, 'Win_Rate'] = (
        len(sponsor_df[sponsor_df['Finish_Position'] == 1]) / total_races * 100
    )#.round(1)

print("\n=== Sponsor Performance Summary ===")
print(sponsor_stats.sort_values('Avg_Finish'))


=== Sponsor Performance Summary ===
             Primary_Sponsor  Avg_Finish  Best_Finish  Finish_StdDev  \
5                Progressive       10.60            1          11.92   
2  Cheddar's Scratch Kitchen       17.50            5           9.85   
0                Busch Light       18.72            6          10.56   
1                    Castrol       18.91            2          13.82   
4              Other/Unknown       19.60            1          10.99   
3        Love's Travel Stops       21.72            2           8.98   

   Total_Laps_Led  Avg_Laps_Led  Num_Drivers  Total_Race_Entries  Top_5_Rate  \
5             839         41.95            1                  20   55.000000   
2              39          3.25            1                  12    8.333333   
0              27          1.08            1                  25    0.000000   
1              39          3.55            1                  11   27.272727   
4           12895          6.73           62              

In [12]:
print("=== Final Validation ===\n")

# 1. Check completeness
expected_races = 53
actual_races = df['race_name'].nunique()
print(f"Race completeness: {actual_races}/{expected_races} ({actual_races/expected_races*100:.0f}%)")

# 2. Check target sponsor coverage
print(f"\nTarget sponsors found: {df[df['Primary_Sponsor'].isin(target_sponsors)]['Primary_Sponsor'].nunique()}")
for sponsor in target_sponsors:
    count = len(df[df['Primary_Sponsor'] == sponsor])
    print(f"  {sponsor}: {count} race entries")

# 3. Check for data quality issues
issues = []

# Unrealistic positions
bad_positions = df[df['Finish_Position'] > 50]
if len(bad_positions) > 0:
    issues.append(f"Positions > 50: {len(bad_positions)}")

# Missing critical fields
for col in ['race_name', 'Driver', 'Owner', 'Finish_Position']:
    missing = df[col].isnull().sum()
    if missing > 0:
        issues.append(f"Missing {col}: {missing}")

# Duplicate entries
duplicates = df.duplicated(subset=['race_name', 'Driver']).sum()
if duplicates > 0:
    issues.append(f"Duplicate entries: {duplicates}")

if issues:
    print("\nData quality issues found:")
    for issue in issues:
        print(f"  - {issue}")
else:
    print("\nNo critical data quality issues found!")

# 4. Summary statistics
print("\n=== Dataset Summary ===")
print(f"Total rows: {len(df)}")
print(f"Unique races: {df['race_name'].nunique()}")
print(f"Unique drivers: {df['Driver'].nunique()}")
print(f"Unique teams: {df['Owner'].nunique()}")
print(f"Unique sponsors: {df['Primary_Sponsor'].nunique()}")
print(f"Date range: {df['Race_Date'].min()} to {df['Race_Date'].max()}")

=== Final Validation ===

Race completeness: 52/53 (98%)

Target sponsors found: 4
  FedEx: 0 race entries
  Castrol: 11 race entries
  Love's Travel Stops: 29 race entries
  Busch Light: 25 race entries
  Cheddar's Scratch Kitchen: 12 race entries

Data quality issues found:
  - Missing Owner: 18
  - Duplicate entries: 36

=== Dataset Summary ===
Total rows: 2013
Unique races: 52
Unique drivers: 62
Unique teams: 20
Unique sponsors: 6
Date range: 2025-02-16 00:00:00 to 2026-06-21 00:00:00


In [13]:
#drop full duplicate rows
df = df.drop_duplicates()


In [14]:
# Save the clean master race results
clean_output = '../csvs/processed/race_results_2025-6_clean.csv'
df.to_csv(clean_output, index=False)
print(f"Saved clean race results to: {clean_output}")

# Save driver statistics
driver_stats_output = '../csvs/processed/driver_stats_2025-6.csv'
driver_stats.to_csv(driver_stats_output, index=False)
print(f"Saved driver statistics to: {driver_stats_output}")

# Save sponsor summary
sponsor_stats_output = '../csvs/processed/sponsor_stats_2025-6.csv'
sponsor_stats.to_csv(sponsor_stats_output, index=False)
print(f"Saved sponsor summary to: {sponsor_stats_output}")

# Save target sponsors dataset
target_output = '../csvs/processed/race_results_targets_2025-6.csv'
df_targets.to_csv(target_output, index=False)
print(f"Saved target sponsors data to: {target_output}")

Saved clean race results to: ../csvs/processed/race_results_2025-6_clean.csv
Saved driver statistics to: ../csvs/processed/driver_stats_2025-6.csv


Saved sponsor summary to: ../csvs/processed/sponsor_stats_2025-6.csv
Saved target sponsors data to: ../csvs/processed/race_results_targets_2025-6.csv
